# THAKAA FLOW - Week 2 Day 4 (Wednesday, 11 Nov 2026)
## EDA + Data Cleaning على بيانات طلبات القروض

## Task 1

### 1. بطاقة الداتا

| اسم العمود | المعنى بكلماتي | النوع | Feature أم Target؟ |
| :--- | :--- | :--- | :--- |
| `Loan_ID` | رقم تعريفي فريد لكل طلب قرض | معرّف (ID) | لا يُستخدم (معرّف فقط) |
| `Gender` | جنس مقدم الطلب (ذكر / أنثى) | تصنيفي | Feature |
| `Married` | الحالة الاجتماعية (متزوج أم لا) | تصنيفي | Feature |
| `Dependents` | عدد الأشخاص الذين يعولهم مقدم الطلب | تصنيفي / رقمي | Feature |
| `Education` | المستوى التعليمي (خريج جامعي أم لا) | تصنيفي | Feature |
| `Self_Employed` | هل يعمل لحسابه الخاص أم موظف | تصنيفي | Feature |
| `ApplicantIncome` | الدخل الشهري لمقدم الطلب الأساسي | رقمي | Feature |
| `CoapplicantIncome` | الدخل الشهري للشريك أو الضامن إن وجد | رقمي | Feature |
| `LoanAmount` | مبلغ القرض المطلوب (بالآلاف) | رقمي | Feature |
| `Loan_Amount_Term` | مدة سداد القرض بالأشهر | رقمي | Feature |
| `Credit_History` | السجل الائتماني (هل التزم بالسداد سابقاً: 1 نعم، 0 لا) | تصنيفي | Feature |
| `Property_Area` | منطقة العقار (حضرية، شبه حضرية، ريفية) | تصنيفي | Feature |
| `Loan_Status` | حالة القرض: هل تمت الموافقة (Y) أم الرفض (N) | تصنيفي | **Target** |

### 2. فرضياتي قبل الأرقام

| العمود (Feature) | توقعي لتأثيره | السبب |
| :--- | :--- | :--- |
| `Credit_History` | يؤثر بقوة | البنك يهمه جداً معرفة إذا كان الشخص ملتزماً بسداد ديونه السابقة أم لا. |
| `ApplicantIncome` | يؤثر بقوة | كلما زاد راتب الشخص زادت قدرته على دفع الأقساط الشهرية. |
| `CoapplicantIncome` | يؤثر قليلاً | وجود دخل إضافي من شريك يدعم الطلب، خاصة لو كان راتب المتقدم الأساسي ضعيفاً. |
| `LoanAmount` | يؤثر بقوة | المبالغ الكبيرة جداً تزيد المخاطرة على البنك مقارنة بالمبالغ الصغيرة. |
| `Loan_Amount_Term` | يؤثر قليلاً | المدة الطويلة تقلل القسط الشهري وتسهل السداد، لكنها تزيد مدة التزام البنك. |
| `Education` | يؤثر قليلاً | الخريج غالباً فرصه الوظيفية واستقرار دخله أعلى. |
| `Married` | يؤثر قليلاً | المتزوجون قد يكون لديهم استقرار أو دخل مشترك، لكن التزاماتهم أيضاً أكبر. |
| `Dependents` | يؤثر قليلاً | كثرة المعالين تزيد مصاريف الشخص وتقلل المبلغ المتبقي لسداد القرض. |
| `Self_Employed` | يؤثر قليلاً | العمل الحر دخله متقلب أحياناً عكس الوظيفة الثابتة. |
| `Property_Area` | يؤثر قليلاً | العقار في المدن قيمته وفرص بيعه أسهل من المناطق الريفية كضمان. |
| `Gender` | لا يؤثر | القدرة على السداد تعتمد على الدخل والالتزام وليس على كون الشخص ذكراً أو أنثى. |

### 3. أهمية معرفة الوحدة قبل التحليل
معرفة الوحدة ضرورية جداً حتى لا نفسر الأرقام بشكل خاطئ ونبني قرارات غير منطقية.
**مثال:** لو لم نعرف أن `LoanAmount` بالآلاف و`Loan_Amount_Term` بالأشهر، قد يظن المحلل أن شخصاً يطلب قرضاً قيمته `150` فقط لمدة `360` سنة! وهذا سيؤدي لحسابات خاطئة تماماً عند مقارنة قيمة القرض براتب العميل الشهري.

### 4. سؤال أخلاقي: استخدام عمود `Gender` في موديل القروض
برأيي ليس من العدل أن يعتمد الموديل على جنس المتقدم لاتخاذ قرار الموافقة أو الرفض. القرار المالي يجب أن يُبنى على قدرة الشخص المالية وسجله الائتماني والتزامه بالسداد، وليس على نوعه. استخدام عمود مثل `Gender` قد يسبب تحيزاً غير عادل ضد فئة معينة لمجرد أن البيانات التاريخية القديمة كان فيها أعداد أقل منهم، والأفضل استبعاده عند بناء الموديل.

## Task 2

In [ ]:
import pandas as pd
import numpy as np

# قراءة الملف
df = pd.read_csv("loan.csv")

# 1. الشكل وأول 3 صفوف ومعلومات الأعمدة
print("Shape:", df.shape)
display(df.head(3))
df.info()

In [ ]:
# 2. جدول القيم الناقصة ونسبتها مرتب تنازلياً
missing_count = df.isna().sum()
missing_pct = (df.isna().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Pct": missing_pct.round(2)
})
missing_df = missing_df[missing_df["Missing_Count"] > 0].sort_values(by="Missing_Count", ascending=False)
display(missing_df)

In [ ]:
# 3. الصفوف المكررة وعدد القيم المختلفة
print("عدد الصفوف المكررة:", df.duplicated().sum())
print("\nعدد القيم المختلفة في كل عمود (nunique):")
print(df.nunique())

In [ ]:
# 4. الإحصاءات الوصفية للأعمدة الرقمية
display(df.describe())

**ملاحظتي على الدخل (`ApplicantIncome`):**
نلاحظ أن المتوسط (`mean`) حوالي 5403 وهو أعلى بكثير من الوسيط (`50%` / `median`) الذي يساوي 3812، بينما القيمة القصوى (`max`) تصل إلى 81000. هذا يوضح أن هناك أشخاصاً دخولهم مرتفعة جداً (Outliers) تسحب المتوسط للأعلى.

In [ ]:
# 5. توزيع القيم للأعمدة التصنيفية شاملة القيم الناقصة
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']
for col in cat_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

**ملاحظتي على عمود `Dependents`:**
نعم، توجد قيمة غريبة وهي `'3+'` (نصية وليست رقماً)، مما جعل العمود يُقرأ كـ `object` بدلاً من رقم.

In [ ]:
# 6. نسبة الطلبات المقبولة والمرفوضة
print("نسبة الطلبات المقبولة والمرفوضة:")
print((df['Loan_Status'].value_counts(normalize=True) * 100).round(2))

### 7. Data Profile وقائمة القرارات المبدئية

| العمود | النوع | نسبة النقص تقريباً | ملاحظتي |
| :--- | :--- | :--- | :--- |
| `Loan_ID` | معرّف | 0% | معرّف فريد لكل صف ولا يوجد تكرار. |
| `Gender` | تصنيفي | ~2.12% | نقص بسيط جداً، وأغلب المتقدمين ذكور. |
| `Married` | تصنيفي | ~0.49% | نقص بسيط جداً (3 صفوف فقط). |
| `Dependents` | تصنيفي | ~2.44% | يحتوي على قيمة `'3+'` ويحتاج تحويل لرقم بعد ملء النقص. |
| `Education` | تصنيفي | 0% | مكتمل ولا يوجد به نقص. |
| `Self_Employed` | تصنيفي | ~5.21% | نقص بسيط، والأغلب غير مستقلين (No). |
| `ApplicantIncome` | رقمي | 0% | مكتمل لكن فيه قيم متطرفة مرتفعة جداً. |
| `CoapplicantIncome` | رقمي | 0% | مكتمل وفيه قيم متطرفة، وكثير من القيم تساوي 0. |
| `LoanAmount` | رقمي | ~3.58% | فيه قيم ناقصة وقيم متطرفة مرتفعة. |
| `Loan_Amount_Term` | رقمي | ~2.28% | أغلب القروض مدتها 360 شهراً (30 سنة). |
| `Credit_History` | تصنيفي (0/1) | ~8.14% | أعلى عمود في نسبة النقص، وهو عمود مهم جداً. |
| `Property_Area` | تصنيفي | 0% | مكتمل (Semiurban, Urban, Rural). |
| `Loan_Status` | Target | 0% | مكتمل: حوالي 68.7% مقبول (Y) و 31.3% مرفوض (N). |

**قائمة القرارات المبدئية للتنفيذ في المهمة 3:**
1. ملء النقص في الأعمدة التصنيفية القليلة (`Gender`, `Married`, `Self_Employed`) بالقيمة الأكثر تكراراً (`mode`).
2. ملء نقص `Dependents` بالـ `mode`، واستبدال `'3+'` بـ `3` وتحويل العمود إلى `int`.
3. ملء نقص `LoanAmount` بالوسيط (`median`) بسبب وجود قيم متطرفة.
4. ملء نقص `Loan_Amount_Term` بالقيمة الأكثر تكراراً (`mode`).
5. فحص نسبة الموافقة في القيم الناقصة لـ `Credit_History` قبل اختيار طريقة الملء.
6. حساب الـ Outliers بطريقة IQR وإبقاؤها لأنها تمثل رواتب وقروض حقيقية وليست أخطاء إدخال.
7. تحويل `Loan_Status` إلى `1` و `0`.

## Task 3

In [ ]:
# نحتفظ بنسخة من الداتا الأصلية للمقارنة في المهمة 4
df_raw = df.copy()

# 1. الأعمدة التصنيفية القليلة النقص: نطبع الـ mode قبل التنفيذ
for col in ['Gender', 'Married', 'Self_Employed']:
    mode_val = df[col].mode()[0]
    missing_before = df[col].isna().sum()
    print(f"{col}: عدد النقص = {missing_before} | الـ mode = {mode_val}")
    df[col] = df[col].fillna(mode_val)

In [ ]:
# 2. عمود Dependents
dep_mode = df['Dependents'].mode()[0]
print("Dependents: عدد النقص =", df['Dependents'].isna().sum(), "| الـ mode =", dep_mode)

df['Dependents'] = df['Dependents'].fillna(dep_mode)
df['Dependents'] = df['Dependents'].replace('3+', 3).astype(int)
print("نوع العمود بعد التحويل:", df['Dependents'].dtype)

In [ ]:
# 3. عمود LoanAmount
loan_mean = df['LoanAmount'].mean()
loan_median = df['LoanAmount'].median()
print(f"LoanAmount -> المتوسط: {loan_mean:.2f} | الوسيط: {loan_median:.2f} | النقص: {df['LoanAmount'].isna().sum()}")

# سنملأ بالوسيط (median) لأن المتوسط متأثر بالقيم المتطرفة المرتفعة
df['LoanAmount'] = df['LoanAmount'].fillna(loan_median)

In [ ]:
# 4. عمود Loan_Amount_Term
term_mode = df['Loan_Amount_Term'].mode()[0]
print(f"Loan_Amount_Term -> الأكثر تكراراً: {term_mode} شهر (يعني {term_mode/12:.0f} سنة) | النقص: {df['Loan_Amount_Term'].isna().sum()}")

df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(term_mode)

In [ ]:
# 5. عمود Credit_History: فحص نسبة الموافقة قبل القرار
temp_ch = df['Credit_History'].fillna(-1)
print("نسبة الموافقة (Y) حسب قيم Credit_History (حيث -1 تعني ناقص):")
print(pd.crosstab(temp_ch, df['Loan_Status'], normalize='index').round(3) * 100)

# النتيجة: الصفوف الناقصة (-1) نسبة الموافقة فيها قريبة جداً من فئة 1 (حوالي 74%) وليست 0 (حوالي 8%).
# لذلك من المنطقي والآمن ملؤها بالقيمة الأكثر تكراراً وهي 1.
ch_mode = df['Credit_History'].mode()[0]
print("\nعدد النقص في Credit_History:", df['Credit_History'].isna().sum(), "| سيتم الملء بـ:", ch_mode)
df['Credit_History'] = df['Credit_History'].fillna(ch_mode)

In [ ]:
# 6. حساب عدد Outliers بطريقة IQR
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: عدد القيم المتطرفة (Outliers) = {len(outliers)}")

In [ ]:
# 7. تحويل Target إلى أرقام
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
print("قيم Loan_Status بعد التحويل:\n", df['Loan_Status'].value_counts())

# 8. التأكد من عدم وجود أي قيم ناقصة وحفظ الملف
print("\nمجموع القيم الناقصة في كامل الجدول:", df.isna().sum().sum())
df.to_csv("loan_clean.csv", index=False)
print("تم حفظ الملف loan_clean.csv بنجاح.")

### Decision Log

| العمود / الإجراء | القرار المتخذ | السبب بالأرقام | عدد الخلايا / الصفوف المتأثرة |
| :--- | :--- | :--- | :--- |
| `Gender` | الملء بـ `'Male'` (`mode`) | القيمة الأكثر تكراراً في العمود (أكثر من 80%). | 13 خلية |
| `Married` | الملء بـ `'Yes'` (`mode`) | القيمة الأكثر تكراراً في العمود. | 3 خلايا |
| `Self_Employed` | الملء بـ `'No'` (`mode`) | القيمة الأكثر تكراراً (حوالي 86% من البيانات). | 32 خلية |
| `Dependents` | الملء بـ `'0'` ثم استبدال `'3+'` بـ `3` وتحويله لـ `int` | `'0'` هي الأكثر تكراراً، وتوحيد `'3+'` يجعل العمود رقمياً قابلاً للتحليل. | 15 خلية ناقصة + 51 خلية كانت `'3+'` |
| `LoanAmount` | الملء بالوسيط (`128.0`) | المتوسط (`146.41`) متأثر بالقيم المتطرفة، فالوسيط أدق. | 22 خلية |
| `Loan_Amount_Term` | الملء بالـ `mode` (`360.0`) | 360 شهراً (30 سنة) هي المدة السائدة لأغلب القروض. | 14 خلية |
| `Credit_History` | الملء بـ `1.0` (`mode`) | نسبة الموافقة في الصفوف الناقصة (~74%) تشبه فئة `1` (~79.6%) ولا تشبه فئة `0` (~7.9%). | 50 خلية |
| Outliers (`IQR`) | الإبقاء عليها وعدم حذفها | تمثل عملاء بدخول أو قروض مرتفعة حقيقية (50 في دخل المتقدم، 18 في الشريك، 41 في القرض). | 0 صفوف محذوفة |
| `Loan_Status` | تحويل `Y` إلى `1` و `N` إلى `0` | لتجهيز عمود الهدف للتحليل الإحصائي والنماذج مستقبلاً. | 614 خلية (كامل العمود) |

## Task 4

In [ ]:
# 1. مقارنة الشكل، القيم الناقصة، ونسبة الموافقة قبل وبعد
print("قبل التنظيف -> Shape:", df_raw.shape, "| القيم الناقصة:", df_raw.isna().sum().sum())
print("بعد التنظيف -> Shape:", df.shape, "| القيم الناقصة:", df.isna().sum().sum())

print("\nنسبة الموافقة قبل التنظيف:")
print((df_raw['Loan_Status'].value_counts(normalize=True) * 100).round(2))
print("\nنسبة الموافقة بعد التنظيف:")
print((df['Loan_Status'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# 2. مقارنة متوسط ووسيط LoanAmount قبل وبعد
print(f"LoanAmount قبل -> المتوسط: {df_raw['LoanAmount'].mean():.2f} | الوسيط: {df_raw['LoanAmount'].median():.2f}")
print(f"LoanAmount بعد -> المتوسط: {df['LoanAmount'].mean():.2f} | الوسيط: {df['LoanAmount'].median():.2f}")

**إجابة السؤال 2:**
الوسيط (`median`) لم يتغير وبقي `128.0` كما هو، بينما المتوسط (`mean`) تغير قليلاً وانخفض من `146.41` إلى `145.75`؛ لأننا أضفنا 22 قيمة جديدة تساوي `128.0` (وهي أقل من المتوسط السابق) مما سحب المتوسط العام للأسفل بشكل طفيف.

In [ ]:
# 3. مقارنة توزيع Credit_History قبل وبعد
print("توزيع Credit_History قبل التنظيف:")
print((df_raw['Credit_History'].value_counts(normalize=True) * 100).round(2))
print("\nتوزيع Credit_History بعد التنظيف:")
print((df['Credit_History'].value_counts(normalize=True) * 100).round(2))

### 4. رسالة لمدير المخاطر
قمنا بفحص بيانات طلبات القروض (614 طلباً) ووجدنا أن المشكلة الأساسية هي وجود بعض الخانات الفارغة في معلومات العملاء، خاصة في السجل الائتماني وقيمة القرض، بالإضافة إلى وجود رواتب وقروض مرتفعة لكنها حقيقية. قمنا بمعالجة جميع الخانات الفارغة باستخدام القيم الأكثر شيوعاً والوسيطة دون حذف أي طلب، وحافظنا على عدد الطلبات ونسبة الموافقة العامة (68.7%) كما هي تماماً دون تشويه. البيانات الآن نظيفة ومكتملة وجاهزة للانتقال لمرحلة بناء النموذج. **الخطر الوحيد الذي يجب الانتباه له** هو أننا افترضنا أن العملاء الذين لم يُسجّل لهم سجل ائتماني (حوالي 8% من الطلبات) لديهم سجل جيد بناءً على سلوكهم، لذا يجب مستقبلاً إلزام الموظفين بإدخال هذه الخانة وعدم تركها فارغة.

## Task 5

In [ ]:
fossils = {
    "fossil_id": ["F01", "F02", "F03", "F04", "F05", "F02", "F06", "F07", "F08", "F09"],
    "species": ["T-Rex", "Triceratops", "t-rex", "Velociraptor", "Stegosaurus", "Triceratops", "Brachiosaurus", "T-Rex", "Velociraptor", "Stegosaurus"],
    "age_million_years": [68, 67, 66, 75, -150, 67, 150, 68, None, 155],
    "length": ["12 m", "9 m", "40 ft", "2 m", "9 m", "9 m", "26 m", "12.3 m", "6.6 ft", "9 m"],
    "found_in": ["USA", "USA", "Canada", "Mongolia", "USA", "USA", "Tanzania", "USA", "Mongolia", "Portugal"],
}
df_fossils = pd.DataFrame(fossils)

# 1. حذف المكرر وتوحيد أسماء الأنواع
df_fossils = df_fossils.drop_duplicates()
df_fossils['species'] = df_fossils['species'].str.strip().str.title()

# 2. تصحيح العمر السالب (-150)
# السبب الأرجح هو خطأ مطبعي (إضافة إشارة سالب بالخطأ أثناء الإدخال)، نصححه بأخذ القيمة المطلقة
df_fossils['age_million_years'] = df_fossils['age_million_years'].abs()

# 3. ملء العمر الناقص بوسيط عمر نفس النوع
df_fossils['age_million_years'] = df_fossils['age_million_years'].fillna(
    df_fossils.groupby('species')['age_million_years'].transform('median')
)

# 4. فصل الطول وتحويله إلى متر في عمود Length_m
split_len = df_fossils['length'].str.split(" ", expand=True)
val = split_len[0].astype(float)
unit = split_len[1]

df_fossils['Length_m'] = np.where(unit == 'ft', val * 0.3048, val).round(2)

display(df_fossils)

In [ ]:
# 5. التحليل المطلوب
print("متوسط طول كل نوع (مرتباً تنازلياً):")
print(df_fossils.groupby('species')['Length_m'].mean().sort_values(ascending=False).round(2))

oldest = df_fossils.loc[df_fossils['age_million_years'].idxmax()]
print(f"\nأقدم نوع: {oldest['species']} بعمر {oldest['age_million_years']} مليون سنة")

top_country = df_fossils['found_in'].value_counts().idxmax()
print(f"الدولة التي وجد فيها أكثر عدد: {top_country}")

**إجابة السؤال 3:**
ملء العمر الناقص بوسيط نفس النوع أدق لأن أحجام وأعمار أنواع الديناصورات تختلف بملايين السنين؛ فمثلاً `Brachiosaurus` عاش قبل 150 مليون سنة بينما `Velociraptor` عاش قبل حوالي 75 مليون سنة، فلو أخذنا وسيط كل الحفريات سنعطي الديناصور عمراً غير منطقي لفصيلته.

### 6. تفكير
* **المشكلة المشابهة لداتا القروض:** وجود قيم نصية غير موحدة ورموز داخل أعمدة يفترض أن تكون أرقاماً موحدة (مثل `'3+'` في عدد المعالين اليوم، واختلاف وحدات القياس `m` و `ft` أو كتابة `T-Rex` و `t-rex` في بيانات المتحف)، بالإضافة إلى ترك خانات فارغة.
* **كيف أمنعها من البداية في نظام الإدخال:** أجعل الإدخال مقيداً بقوائم منسدلة (Dropdown lists) للأسماء والوحدات بدلاً من الكتابة الحرة، وأضع شروط تحقق (Validation rules) تمنع حفظ النموذج إذا كان هناك حقل إجباري فارغ أو رقم سالب غير منطقي.